In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
%matplotlib inline

In [80]:
def summarize_params(df, groupby=["method", "dataset", "params"], reduce_params=True):
    mid_res = [[n[0], n[1], n[2], g["valid_auc"].mean(), g["auc"].mean(),  g["auc"].std(), g["accuracy"].mean(), g["accuracy"].std(), g["f1"].mean(), g["f1"].std()] for n, g in df.groupby(by=groupby)]
    mid_res = pd.DataFrame(mid_res, columns=["method", 'dataset', 'params', 'valid_auc', 'auc', 'auc_std', 'f1', 'f1_std', 'acc', 'acc_std'])
    params = mid_res["params"]
    mid_res = mid_res.drop("params", axis=1)
    mid_res["params"] = params
    return mid_res

In [81]:
path = "/nfs/wangyu/TGAT-bk/result/"
dirs = os.listdir( path )

# 输出所有文件和文件夹
for file in dirs:
   print(file)

.ipynb_checkpoints
ia-contact-tgat.csv
ia-escorts-dynamic-tgat.csv
ia-contacts_hypertext2009-tgat.csv
fb-forum-tgat.csv
ia-movielens-user2tags-10m-tgat.csv
ia-enron-employees-tgat.csv
ia-workplace-contacts-tgat.csv
ia-retweet-pol-tgat.csv
soc-wiki-elec-tgat.csv
ia-slashdot-reply-dir-tgat.csv
ia-primary-school-proximity-tgat.csv
ia-radoslaw-email-tgat.csv
ia-frwikinews-user-edits-tgat.csv
soc-sign-bitcoinotc-tgat.csv
JODIE-wikipedia-tgat.csv
JODIE-mooc-tgat.csv
JODIE-reddit-tgat.csv
JODIE-lastfm-tgat.csv


In [82]:
import re
path = "/nfs/wangyu/TGAT-bk/result/"
tgat = pd.concat([pd.read_csv(path+file) for file in os.listdir(path) if file.endswith('csv')])
print(tgat.info())
tgat.head()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 92 entries, 0 to 4
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   method     92 non-null     object 
 1   dataset    92 non-null     object 
 2   valid_auc  92 non-null     float64
 3   accuracy   92 non-null     float64
 4   f1         92 non-null     float64
 5   auc        92 non-null     float64
 6   params     92 non-null     object 
dtypes: float64(4), object(3)
memory usage: 5.8+ KB
None


,method,dataset,valid_auc,accuracy,f1,auc,params
0,tgat,ia-contact,0.9053,0.8807,0.8849,0.9148,"n_layer=2,n_head=2,time=time,freeze=False,unif..."
1,tgat,ia-contact,0.8534,0.8368,0.8509,0.8835,"n_layer=2,n_head=2,time=time,freeze=False,unif..."
2,tgat,ia-contact,0.8502,0.8221,0.8420,0.8735,"n_layer=2,n_head=2,time=time,freeze=True,unifo..."
3,tgat,ia-contact,0.9068,0.8855,0.8907,0.9201,"n_layer=2,n_head=2,time=time,freeze=True,unifo..."
4,tgat,ia-contact,0.8962,0.8375,0.8522,0.8943,"n_layer=2,n_head=2,time=empty,freeze=False,uni..."


In [83]:
tgat['params'] = tgat['params'].apply(lambda s: re.sub(r'n_layer=2,n_head=2,', '', s))
sorted(tgat['params'].unique())

['time=empty,freeze=False,uniform=False',
 'time=time,freeze=False,uniform=False',
 'time=time,freeze=False,uniform=True',
 'time=time,freeze=True,uniform=False',
 'time=time,freeze=True,uniform=True']

In [84]:
mid_res = summarize_params(tgat)
mid_res.head()

,method,dataset,valid_auc,auc,auc_std,f1,f1_std,acc,acc_std,params
0,tgat,JODIE-lastfm,0.8709,0.8531,NaN,0.7706,NaN,0.7697,NaN,"time=empty,freeze=False,uniform=False"
1,tgat,JODIE-lastfm,0.8786,0.8659,NaN,0.7810,NaN,0.7832,NaN,"time=time,freeze=False,uniform=False"
2,tgat,JODIE-lastfm,0.7977,0.7719,NaN,0.7014,NaN,0.6889,NaN,"time=time,freeze=False,uniform=True"
3,tgat,JODIE-lastfm,0.5874,0.5805,NaN,0.5447,NaN,0.5873,NaN,"time=time,freeze=True,uniform=False"
4,tgat,JODIE-lastfm,0.5131,0.5060,NaN,0.5042,NaN,0.5949,NaN,"time=time,freeze=True,uniform=True"


In [85]:
comp_param = {
    'time=empty,freeze=False,uniform=False': 'Empty,F,Temporal',
    'time=time,freeze=False,uniform=False': 'Time,F,Temporal',
    'time=time,freeze=False,uniform=True': 'Time,F,Uniform',
    'time=time,freeze=True,uniform=False': 'Time,T,Temporal',
    'time=time,freeze=True,uniform=True': 'Time,T,Uniform'
}
tmp = mid_res[mid_res['params'].isin(comp_param)].copy()
tmp['params'] = tmp['params'].map(comp_param)
# tmp.pivot(index='dataset', columns='params')
p = tmp.pivot(index='dataset', columns='params')

In [86]:
p['auc']

params,"Empty,F,Temporal","Time,F,Temporal","Time,F,Uniform","Time,T,Temporal","Time,T,Uniform"
dataset,,,,,
JODIE-lastfm,0.8531,0.8659,0.7719,0.5805,0.50600
JODIE-mooc,0.8767,0.9046,0.8514,0.7911,0.74080
JODIE-reddit,0.9883,0.9878,0.9874,0.9825,0.98390
JODIE-wikipedia,0.9779,0.9778,0.9773,0.9632,0.96660
fb-forum,0.8878,0.8743,0.8670,0.7823,0.78210
ia-contact,0.8943,0.9148,0.8835,0.9201,0.87350
ia-contacts_hypertext2009,0.9101,0.9523,0.9016,0.9377,0.75160
ia-enron-employees,0.9063,0.8734,0.8819,0.6613,0.67530
ia-escorts-dynamic,0.9099,0.9350,0.9353,0.9431,0.93960


In [11]:
# import torch
# cuda = torch.device('cuda')
# x = torch.randn((1, 1), requires_grad=True)
# with torch.autograd.profiler.profile(use_cuda=True) as prof:
# #     for _ in range(100):  # any normal python code, really!
#     y = x ** 2
#     y.backward()
# print(prof)
# # print(prof.key_averages().table(sort_by="self_cpu_time_total"))

from torch.autograd import Variable
import torch
x = Variable(torch.randn(1, 1), requires_grad=True)
with torch.autograd.profiler.profile(use_cuda=True) as prof:
     y = x ** 2
     y.backward()
# NOTE: some columns were removed for brevity
print(prof)
print(prof.key_averages().table(sort_by="self_cpu_time_total"))

-----------------------------------  ---------------  ---------------  ---------------  ---------------  ---------------  ---------------  ---------------  ---------------  ---------------  -----------------------------------  
Name                                 Self CPU total %  Self CPU total   CPU total %      CPU total        CPU time avg     CUDA total %     CUDA total       CUDA time avg    Number of Calls  Input Shapes                         
-----------------------------------  ---------------  ---------------  ---------------  ---------------  ---------------  ---------------  ---------------  ---------------  ---------------  -----------------------------------  
pow                                  29.23%           121.791us        29.23%           121.791us        121.791us        28.48%           113.662us        113.662us        1                []                                   
torch::autograd::GraphRoot           4.61%            19.224us         4.61%           